# Step 16: Employee Intelligence Master Table (Core Business Output)

## Overview
This notebook combines all workforce analytics into a single master tabular dataset (one row per employee):
- Predicted Attrition Risk Probability (%) & Risk Tier (**HIGH**, **MEDIUM**, **LOW**).
- Engagement, Satisfaction, and Work-Life Balance scores.
- Role & O*NET SOC taxonomy metadata.
- Total missing skill count & weighted skill gap score.
- Top recommended upskilling course path.
- Export to `data/processed/employee_intelligence_master.csv`.


In [1]:
import pandas as pd
import numpy as np
import os
import joblib

PROCESSED_DIR = os.path.join("..", "data", "processed")
MODELS_DIR = os.path.join("..", "models")

attr_df = pd.read_csv(os.path.join(PROCESSED_DIR, "employee_attrition_processed.csv"))
features_df = pd.read_csv(os.path.join(PROCESSED_DIR, "features_engineered.csv"))
role_master = pd.read_csv(os.path.join(PROCESSED_DIR, "role_intelligence_master.csv"))
gap_summary = pd.read_csv(os.path.join(PROCESSED_DIR, "employee_skill_gaps_summary.csv"))
recs_df = pd.read_csv(os.path.join(PROCESSED_DIR, "upskilling_recommendations.csv"))

pipeline = joblib.load(os.path.join(MODELS_DIR, "attrition_pipeline.joblib"))

print(f"Total employees: {len(attr_df)}")


Total employees: 1470


---
## 1. Predict Attrition Probabilities & Risk Tiers


In [2]:
# Prepare features for prediction
drop_cols = ['EmployeeNumber', 'Employee ID', 'Attrition', 'Target_Attrition']
feature_cols = [c for c in features_df.columns if c not in drop_cols]
X = features_df[feature_cols]

attr_probs = pipeline.predict_proba(X)[:, 1]
features_df['Attrition_Probability'] = np.round(attr_probs, 4)

def assign_risk_tier(prob):
    if prob >= 0.50:
        return 'HIGH'
    elif prob >= 0.30:
        return 'MEDIUM'
    else:
        return 'LOW'

features_df['Attrition_Risk_Tier'] = features_df['Attrition_Probability'].apply(assign_risk_tier)

print("=== Attrition Risk Tier Distribution ===")
print(features_df['Attrition_Risk_Tier'].value_counts())


=== Attrition Risk Tier Distribution ===
Attrition_Risk_Tier
LOW       770
HIGH      439
MEDIUM    261
Name: count, dtype: int64


---
## 2. Combine Role, Skills, Gaps, and Course Recommendations


In [3]:
# Get primary recommended course per employee (highest similarity match)
top_recs = recs_df.sort_values(by='Match_Similarity', ascending=False).groupby('EmployeeNumber').first().reset_index()

# Merge onto features_df
master_df = pd.merge(features_df, gap_summary, on='EmployeeNumber', how='left')
master_df = pd.merge(master_df, top_recs[['EmployeeNumber', 'Recommended_Course_Title', 'Recommended_Course_ID']], on='EmployeeNumber', how='left')
master_df = pd.merge(master_df, role_master[['HR_Job_Role', 'ONET_SOC_Code', 'ONET_Title']], left_on='JobRole', right_on='HR_Job_Role', how='left')

# Fill defaults
master_df['Missing_Skills_Count'] = master_df['Missing_Skills_Count'].fillna(0).astype(int)
master_df['Total_Weighted_Gap_Score'] = master_df['Total_Weighted_Gap_Score'].fillna(0.0)
master_df['Recommended_Course_Title'] = master_df['Recommended_Course_Title'].fillna('Leadership Development Core')

print(f"Final Employee Intelligence Master Shape: {master_df.shape}")
assert len(master_df) == 1470, f"Expected 1,470 rows, got {len(master_df)}"

print("\nSample Master Dataset Output (Top 5 High-Risk Employees):")
print(master_df[master_df['Attrition_Risk_Tier']=='HIGH'][['EmployeeNumber', 'Department', 'JobRole', 'MonthlyIncome', 'Attrition_Probability', 'Attrition_Risk_Tier', 'Missing_Skills_Count', 'Recommended_Course_Title']].head(5).to_string(index=False))

out_path = os.path.join(PROCESSED_DIR, "employee_intelligence_master.csv")
master_df.to_csv(out_path, index=False)
print(f"\nSaved Core Master Dataset: {out_path}")


Final Employee Intelligence Master Shape: (1470, 51)

Sample Master Dataset Output (Top 5 High-Risk Employees):
 EmployeeNumber             Department               JobRole  MonthlyIncome  Attrition_Probability Attrition_Risk_Tier  Missing_Skills_Count                      Recommended_Course_Title
              1                  Sales       Sales Executive           5993                 0.9013                HIGH                    52   Enterprise Data Analytics & Microsoft Excel
              4 Research & Development Laboratory Technician           2090                 0.7985                HIGH                    14   Enterprise Data Analytics & Microsoft Excel
              7 Research & Development Laboratory Technician           3468                 0.8186                HIGH                    14 Critical Thinking & Strategic Problem Solving
             15 Research & Development Laboratory Technician           4193                 0.6695                HIGH                    14